# 02 - Evaluate all pruned models on all benchmarks

Given the three calibration artifacts produced by `01_prune_all.ipynb`, this
notebook launches **18 EC2 spot GPU boxes simultaneously** (9 freeform eval +
9 teacher-forced) covering every combination of calibration dataset × eval
benchmark:

```
calibration \ eval  │ GSM8K  │ HumanEval+  │ ARC-Challenge
────────────────────┼────────┼─────────────┼──────────────
GSM8K               │   ✓    │      ✓      │      ✓
HumanEval+          │   ✓    │      ✓      │      ✓
ARC-Challenge       │   ✓    │      ✓      │      ✓
```

Each job runs all five pruning levels [0, 20, 40, 60, 80]%.

**Freeform eval** measures accuracy (pass@1 / exact-match) and teacher-forced
perplexity of the ground-truth answer at each pruning level.

**Teacher-forced collection** scores every test record's ground-truth answer
token-by-token for all pruning levels (`NUM_TF_SAMPLES=0` = full test set).

Result URIs are written back to `experiment_config.json` for collection by
`03_collect_results.ipynb`.


In [1]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)
print("AWS_PROFILE =", os.environ.get("AWS_PROFILE"))


REPO_ROOT = /var/home/three-kingdoms/work/pruning-metrics
AWS_PROFILE = rengz


## Load calibration artifact URIs

Reads `experiment_config.json` written by `01_prune_all.ipynb`. If the file
is missing, set `CALIBRATION_ARTIFACT_URIS` manually below.


In [2]:
import json
import boto3

AWS_PROFILE = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get(
    "RESULTS_BUCKET", "pruning-metrics-results-414266451290"
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

NOTEBOOK_DIR = Path(os.getcwd())
experiment_config_path = NOTEBOOK_DIR / "experiment_config.json"

if experiment_config_path.exists():
    _cfg = json.loads(experiment_config_path.read_text(encoding="utf-8"))
    CALIBRATION_ARTIFACT_URIS = _cfg.get("calibration_artifact_uris", {})
else:
    # Fallback: set URIs manually.
    CALIBRATION_ARTIFACT_URIS = {
        "gsm8k":         "s3://<bucket>/pruning_artifacts/<gsm8k_run_id>/",
        "humaneval":     "s3://<bucket>/pruning_artifacts/<humaneval_run_id>/",
        "arc_challenge": "s3://<bucket>/pruning_artifacts/<arc_challenge_run_id>/",
    }

for label, uri in CALIBRATION_ARTIFACT_URIS.items():
    assert uri.startswith("s3://") and "<" not in uri, (
        f"[{label}] URI not set: {uri!r}. Run 01_prune_all.ipynb first."
    )
    print(f"  {label}: {uri}")


  gsm8k: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144103Z-b08812/
  humaneval: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144135Z-991a93/
  arc_challenge: s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144208Z-122c3c/


## Evaluation configuration

`EVAL_SPECS` defines which benchmarks to evaluate each pruned model on. By
default all three benchmarks are evaluated against every calibration artifact,
giving a full 3×3 cross-evaluation matrix.

`NUM_TF_SAMPLES=0` tells the teacher-forced runner to score **all** test
records (not just a random sample), so the TF logit collection is complete.


In [3]:
EVAL_SPECS = [
    ("gsm8k",         "math:gsm8k:main"),
    ("humaneval",     "coding:evalplus/humanevalplus:test"),
    ("arc_challenge", "mcq:allenai/ai2_arc:ARC-Challenge"),
]
EVAL_LEVELS      = [0, 20, 40, 60, 80]
GENERATION_SEED  = 65320
MAX_NEW_TOKENS   = 512
TIMEOUT_SECONDS  = 10.0
MAX_TEST_SAMPLES = 0    # 0 = all test records
NUM_TF_SAMPLES   = 0    # 0 = all test records for teacher forcing
TF_TOP_K         = 5

INSTANCE_TYPE_PRIORITY = ["p4de.24xlarge", "p5.48xlarge", "p4d.24xlarge"]
REGION_PRIORITY        = ["us-east-1", "us-west-2", "us-east-2"]

print(json.dumps({
    "EVAL_SPECS": EVAL_SPECS,
    "EVAL_LEVELS": EVAL_LEVELS,
    "NUM_TF_SAMPLES": NUM_TF_SAMPLES,
    "TF_TOP_K": TF_TOP_K,
}, indent=2))


{
  "EVAL_SPECS": [
    [
      "gsm8k",
      "math:gsm8k:main"
    ],
    [
      "humaneval",
      "coding:evalplus/humanevalplus:test"
    ],
    [
      "arc_challenge",
      "mcq:allenai/ai2_arc:ARC-Challenge"
    ]
  ],
  "EVAL_LEVELS": [
    0,
    20,
    40,
    60,
    80
  ],
  "NUM_TF_SAMPLES": 0,
  "TF_TOP_K": 5
}


## Find spot capacity

Queries available spot instances. All 18 EC2 jobs use the same first
candidate. If `InsufficientInstanceCapacity` occurs for any launch, re-run
that individual `launch_runner` call with `candidates[1]`.


In [4]:
from pruning_metrics.notebook_helpers import find_capacity

candidates = find_capacity(
    regions=tuple(REGION_PRIORITY),
    instance_types=tuple(INSTANCE_TYPE_PRIORITY),
    aws_profile=AWS_PROFILE,
)
assert candidates, "No spot capacity found."
chosen = candidates[0]
print(f"Chosen: {chosen['region']} {chosen['availability_zone']} "
      f"{chosen['instance_type']} @ ${chosen['spot_price_usd_per_hour']:.4f}/h")


Chosen: us-east-1 us-east-1d p4de.24xlarge @ $12.8718/h


## Launch all 9 freeform eval runners

One EC2 job per `(calibration_dataset, eval_benchmark)` pair. Each job
evaluates the pruned model at all five sparsity levels and computes both
accuracy and teacher-forced perplexity on the ground-truth answers.


In [5]:
from pruning_metrics.notebook_helpers import launch_runner, render_run_id_default

freeform_launches = []  # list of (cal_label, eval_label, LaunchedRun)

for cal_label, cal_uri in CALIBRATION_ARTIFACT_URIS.items():
    for eval_label, eval_spec in EVAL_SPECS:
        run_id = render_run_id_default()
        runner_env = {
            "PRUNING_ARTIFACT_URI": cal_uri,
            "EVAL_DATASET_SPEC": eval_spec,
            "EVAL_LEVELS": ",".join(str(lv) for lv in EVAL_LEVELS),
            "GENERATION_SEED": GENERATION_SEED,
            "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
            "TIMEOUT_SECONDS": TIMEOUT_SECONDS,
            "MAX_TEST_SAMPLES": MAX_TEST_SAMPLES,
        }
        launched = launch_runner(
            runner="freeform_eval",
            runner_env=runner_env,
            region=chosen["region"],
            availability_zone=chosen["availability_zone"],
            instance_type=chosen["instance_type"],
            max_spot_price=float(chosen["max_bid_usd_per_hour"]),
            results_bucket=RESULTS_BUCKET,
            results_prefix="freeform_eval",
            run_id=run_id,
            aws_profile=AWS_PROFILE,
            hf_token=HF_TOKEN,
            name_tag=f"pm-freeform-{cal_label}-{eval_label}",
        )
        freeform_launches.append((cal_label, eval_label, launched))
        print(
            f"[{cal_label} -> {eval_label}] {launched.instance_id} "
            f"run_id={launched.run_id}"
        )

print(f"\nLaunched {len(freeform_launches)} freeform eval runners.")


CalledProcessError: Command '['/var/home/three-kingdoms/work/pruning-metrics/.venv/bin/python', '/var/home/three-kingdoms/work/pruning-metrics/infra/ec2/launch_gpu_instance.py', '--region', 'us-east-1', '--availability-zone', 'us-east-1d', '--instance-type', 'p4de.24xlarge', '--max-spot-price', '32.1795', '--results-bucket', 'pruning-metrics-results-414266451290', '--results-prefix', 'freeform_eval', '--instance-profile', 'pruning-metrics-ec2', '--name-tag', 'pm-freeform-gsm8k-gsm8k', '--runner', 'freeform_eval', '--runner-env-json', '{"PRUNING_ARTIFACT_URI": "s3://pruning-metrics-results-414266451290/pruning_artifacts/20260506T144103Z-b08812/", "EVAL_DATASET_SPEC": "math:gsm8k:main", "EVAL_LEVELS": "0,20,40,60,80", "GENERATION_SEED": "65320", "MAX_NEW_TOKENS": "512", "TIMEOUT_SECONDS": "10.0", "MAX_TEST_SAMPLES": "0"}', '--run-id', '20260506T154106Z-162ef8']' returned non-zero exit status 1.

## Wait for all freeform eval runs to finish

Polls `summary.json` (with `ended_at_utc` set) or instance termination for
each of the 9 jobs in parallel. Typical wall-clock time per job:
~3–5 h for Qwen2-72B over 5 pruning levels on a p4de.24xlarge.


In [ ]:
import concurrent.futures
import time

from pruning_metrics.notebook_helpers import wait_for_runner_completion

def _wait_freeform(cal_label, eval_label, launched):
    summary_key = f"freeform_eval/{launched.run_id}/summary.json"
    reason, summary = wait_for_runner_completion(
        bucket=RESULTS_BUCKET,
        summary_key=summary_key,
        instance_id=launched.instance_id,
        region=launched.region,
        aws_profile=AWS_PROFILE,
        poll_seconds=60.0,
        timeout_seconds=60 * 60 * 8,
        progress_log_interval_seconds=600.0,
    )
    completed_levels = (summary or {}).get("completed_levels", [])
    print(
        f"[{cal_label} -> {eval_label}] {reason} "
        f"levels_done={completed_levels}"
    )
    return reason, summary

t0 = time.monotonic()
freeform_results = {}  # (cal_label, eval_label) -> (reason, summary)

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(freeform_launches)
) as pool:
    futs = {
        pool.submit(_wait_freeform, cal, ev, launched): (cal, ev)
        for cal, ev, launched in freeform_launches
    }
    for fut in concurrent.futures.as_completed(futs):
        cal, ev = futs[fut]
        try:
            freeform_results[(cal, ev)] = fut.result()
        except Exception as exc:
            print(f"[{cal} -> {ev}] ERROR: {exc}")
            freeform_results[(cal, ev)] = (str(exc), None)

elapsed = time.monotonic() - t0
print(f"\nAll freeform evals complete in {elapsed / 60:.1f} min.")


## Launch all 9 teacher-forced runners

Same 3×3 matrix. `NUM_TF_SAMPLES=0` instructs the runner to score every
record in the test split (1 319 for GSM8K, 1 172 for ARC-Challenge, ~33 for
HumanEval+), producing per-token log-probabilities and top-5 alternatives for
all pruning levels.


In [ ]:
tf_launches = []  # list of (cal_label, eval_label, LaunchedRun)

for cal_label, cal_uri in CALIBRATION_ARTIFACT_URIS.items():
    for eval_label, eval_spec in EVAL_SPECS:
        run_id = render_run_id_default()
        runner_env = {
            "PRUNING_ARTIFACT_URI": cal_uri,
            "EVAL_DATASET_SPEC": eval_spec,
            "EVAL_LEVELS": ",".join(str(lv) for lv in EVAL_LEVELS),
            "NUM_TF_SAMPLES": NUM_TF_SAMPLES,
            "TF_TOP_K": TF_TOP_K,
        }
        launched = launch_runner(
            runner="teacher_forced",
            runner_env=runner_env,
            region=chosen["region"],
            availability_zone=chosen["availability_zone"],
            instance_type=chosen["instance_type"],
            max_spot_price=float(chosen["max_bid_usd_per_hour"]),
            results_bucket=RESULTS_BUCKET,
            results_prefix="teacher_forced",
            run_id=run_id,
            aws_profile=AWS_PROFILE,
            hf_token=HF_TOKEN,
            name_tag=f"pm-tf-{cal_label}-{eval_label}",
        )
        tf_launches.append((cal_label, eval_label, launched))
        print(
            f"[{cal_label} -> {eval_label}] {launched.instance_id} "
            f"run_id={launched.run_id}"
        )

print(f"\nLaunched {len(tf_launches)} teacher-forced runners.")


## Wait for all teacher-forced runs to finish

Typical wall-clock time: ~4–6 h per job for the full GSM8K test set (1 319
records × 5 levels × one forward pass each) on a p4de.24xlarge. All 9 jobs
run simultaneously.


In [ ]:
def _wait_tf(cal_label, eval_label, launched):
    summary_key = f"teacher_forced/{launched.run_id}/summary.json"
    reason, summary = wait_for_runner_completion(
        bucket=RESULTS_BUCKET,
        summary_key=summary_key,
        instance_id=launched.instance_id,
        region=launched.region,
        aws_profile=AWS_PROFILE,
        poll_seconds=60.0,
        timeout_seconds=60 * 60 * 10,
        progress_log_interval_seconds=600.0,
    )
    num_samples = len((summary or {}).get("samples", {}))
    print(
        f"[{cal_label} -> {eval_label}] {reason} samples_done={num_samples}"
    )
    return reason, summary

t0 = time.monotonic()
tf_results = {}  # (cal_label, eval_label) -> (reason, summary)

with concurrent.futures.ThreadPoolExecutor(max_workers=len(tf_launches)) as pool:
    futs = {
        pool.submit(_wait_tf, cal, ev, launched): (cal, ev)
        for cal, ev, launched in tf_launches
    }
    for fut in concurrent.futures.as_completed(futs):
        cal, ev = futs[fut]
        try:
            tf_results[(cal, ev)] = fut.result()
        except Exception as exc:
            print(f"[{cal} -> {ev}] ERROR: {exc}")
            tf_results[(cal, ev)] = (str(exc), None)

elapsed = time.monotonic() - t0
print(f"\nAll teacher-forced runs complete in {elapsed / 60:.1f} min.")


## Persist all result URIs

Writes freeform eval and teacher-forced S3 URIs to `experiment_config.json`
so `03_collect_results.ipynb` can download the summaries without re-running
this notebook.


In [ ]:
freeform_eval_uris = {
    f"{cal}_{ev}": launched.results_uri
    for cal, ev, launched in freeform_launches
}
teacher_forced_uris = {
    f"{cal}_{ev}": launched.results_uri
    for cal, ev, launched in tf_launches
}

# Refresh config from disk (may have been updated elsewhere).
if experiment_config_path.exists():
    cfg = json.loads(experiment_config_path.read_text(encoding="utf-8"))
else:
    cfg = {}

cfg["freeform_eval_uris"]   = freeform_eval_uris
cfg["teacher_forced_uris"]  = teacher_forced_uris
# Also store launch metadata for provenance.
cfg["freeform_launches"] = [
    {"cal": cal, "eval": ev, "run_id": launched.run_id,
     "instance_id": launched.instance_id, "uri": launched.results_uri}
    for cal, ev, launched in freeform_launches
]
cfg["tf_launches"] = [
    {"cal": cal, "eval": ev, "run_id": launched.run_id,
     "instance_id": launched.instance_id, "uri": launched.results_uri}
    for cal, ev, launched in tf_launches
]

experiment_config_path.write_text(
    json.dumps(cfg, indent=2), encoding="utf-8"
)

print("Updated", experiment_config_path)
print("\nFreeform eval URIs:")
for k, v in freeform_eval_uris.items():
    print(f"  {k}: {v}")
print("\nTeacher-forced URIs:")
for k, v in teacher_forced_uris.items():
    print(f"  {k}: {v}")
